In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

print("Project environment ready.")

Project environment ready.


# Financial Services Data Quality & Governance Framework

## Portfolio Disclaimer
This project uses entirely synthetic data generated solely for educational and portfolio purposes.

It does not contain proprietary, confidential, production, or employer-owned data, code, schemas, screenshots, or internal business rules from any organization.

## Project Objective
The goal of this project is to demonstrate practical skills in:

- Python data analysis
- SQL
- Data quality monitoring
- Data governance
- Critical data element validation
- Data remediation
- Exploratory data analysis
- Data visualization
- Financial-services analytics

## 1. Load and Inspect the Synthetic Datasets

We begin by loading the synthetic source and reporting datasets and reviewing their structure, size, data types, and basic data-quality characteristics.

In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

source_df = pd.read_csv(
    RAW_DATA_DIR / "source_financial_data.csv",
    parse_dates=["origination_date", "maturity_date", "as_of_date"]
)

reporting_df = pd.read_csv(
    RAW_DATA_DIR / "reporting_financial_data.csv",
    parse_dates=["origination_date", "maturity_date", "as_of_date"]
)

issue_truth_df = pd.read_csv(
    PROCESSED_DATA_DIR / "injected_issue_truth.csv"
)

print(f"Source dataset shape: {source_df.shape}")
print(f"Reporting dataset shape: {reporting_df.shape}")
print(f"Known injected issues: {len(issue_truth_df):,}")

Source dataset shape: (10000, 12)
Reporting dataset shape: (10050, 12)
Known injected issues: 670


In [3]:
source_df.head()

,record_id,customer_id,product_type,product_subtype,customer_segment,currency,position_amount,origination_date,maturity_date,status,source_system,as_of_date
0,FS-000001,CUST-00580,Commitments,Credit Line,Commercial,USD,66775.13,2024-02-08,2026-11-23,Active,DepositPlatform,2026-10-01
1,FS-000002,CUST-00869,Deposits,Term Deposit,Institutional,USD,2213.38,2019-02-16,2023-07-22,Active,LendingHub,2026-10-01
2,FS-000003,CUST-02153,Securities,Corporate Bond,Consumer,JPY,87162.80,2025-12-21,2030-04-04,Closed,InvestmentPlatform,2026-10-01
3,FS-000004,CUST-00258,Commitments,Standby Commitment,Consumer,USD,29477.56,2020-04-19,2020-09-09,Pending,LendingHub,2026-10-01
4,FS-000005,CUST-00272,Loans,Revolving Loan,Commercial,USD,11592.23,2018-02-01,2023-02-27,Active,LendingHub,2026-10-01


In [4]:
reporting_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10050 entries, 0 to 10049
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   record_id         10050 non-null  str           
 1   customer_id       9950 non-null   str           
 2   product_type      10050 non-null  str           
 3   product_subtype   10050 non-null  str           
 4   customer_segment  10050 non-null  str           
 5   currency          10050 non-null  str           
 6   position_amount   10050 non-null  float64       
 7   origination_date  10050 non-null  datetime64[us]
 8   maturity_date     8573 non-null   datetime64[us]
 9   status            10050 non-null  str           
 10  source_system     10050 non-null  str           
 11  as_of_date        10050 non-null  datetime64[us]
dtypes: datetime64[us](3), float64(1), str(8)
memory usage: 1.6 MB


In [5]:
reporting_df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
record_id,10050,10000,FS-000058,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,9950,2439,CUST-00015,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_type,10050,5,Loans,3537,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_subtype,10050,8,Revolving Loan,1801,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_segment,10050,4,Commercial,3544,NaN,NaN,NaN,NaN,NaN,NaN,NaN
currency,10050,6,USD,6947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
position_amount,10050.0,NaN,NaN,NaN,57768.584622,-448219.11,17975.2975,34905.72,69216.9825,5428835.61,92705.544579
origination_date,10050,NaN,NaN,NaN,2022-05-18 11:56:59.462686,2018-01-01 00:00:00,2020-03-05 00:00:00,2022-05-21 12:00:00,2024-07-19 00:00:00,2026-09-30 00:00:00,NaN
maturity_date,8573,NaN,NaN,NaN,2027-06-03 20:32:53.638166,2017-12-14 00:00:00,2024-08-10 00:00:00,2027-06-07 00:00:00,2030-02-22 00:00:00,2036-08-09 00:00:00,NaN
status,10050,3,Active,7883,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Portfolio Distribution by Product Type

Understanding the composition of the portfolio provides context for subsequent data-quality analysis and helps identify which product groups contribute the greatest record volume.

In [6]:
product_distribution = (
    reporting_df["product_type"]
    .value_counts()
    .reset_index()
)

product_distribution.columns = [
    "product_type",
    "record_count"
]

product_distribution

,product_type,record_count
0,Loans,3537
1,Deposits,3008
2,Securities,1954
3,Commitments,1501
4,UNKNOWN,50


In [7]:
fig = px.bar(
    product_distribution,
    x="product_type",
    y="record_count",
    title="Reporting Records by Product Type",
    labels={
        "product_type": "Product Type",
        "record_count": "Number of Records"
    }
)

fig.show()

## 3. Missing Data Profile

Missing values are evaluated across the reporting dataset to identify completeness issues and distinguish expected nulls from potential data-quality defects.

In [8]:
missing_profile = (
    reporting_df.isna()
    .sum()
    .reset_index()
)

missing_profile.columns = [
    "field",
    "missing_count"
]

missing_profile["missing_percentage"] = (
    missing_profile["missing_count"]
    / len(reporting_df)
    * 100
).round(2)

missing_profile = (
    missing_profile[
        missing_profile["missing_count"] > 0
    ]
    .sort_values(
        "missing_count",
        ascending=False
    )
)

missing_profile

,field,missing_count,missing_percentage
8,maturity_date,1477,14.7
1,customer_id,100,1.0


In [9]:
fig = px.bar(
    missing_profile,
    x="field",
    y="missing_count",
    title="Missing Values by Field",
    labels={
        "field": "Field",
        "missing_count": "Missing Records"
    },
    hover_data=["missing_percentage"]
)

fig.show()

## 4. Rule-Based Data Quality Validation

The reporting dataset is evaluated against a set of generic financial-services data-quality rules covering completeness, validity, accuracy, consistency, timeliness, and uniqueness.

Not every null value represents a defect. Business context is incorporated into the validation logic.

In [11]:
VALID_PRODUCTS = {
    "Loans",
    "Deposits",
    "Commitments",
    "Securities"
}

VALID_CURRENCIES = {
    "USD",
    "EUR",
    "GBP",
    "CAD",
    "JPY"
}

EXPECTED_AS_OF_DATE = source_df["as_of_date"].max()

dq_summary = []

def add_check(rule_name, dimension, failed_count):
    dq_summary.append(
        {
            "rule": rule_name,
            "dimension": dimension,
            "failed_records": int(failed_count),
            "failure_rate_pct": round(
                failed_count / len(reporting_df) * 100,
                2
            )
        }
    )

In [12]:
# 1. Missing Customer ID
missing_customer = reporting_df["customer_id"].isna()

add_check(
    "Missing Customer ID",
    "Completeness",
    missing_customer.sum()
)

# 2. Invalid Currency
invalid_currency = ~reporting_df["currency"].isin(VALID_CURRENCIES)

add_check(
    "Invalid Currency",
    "Validity",
    invalid_currency.sum()
)

# 3. Negative Position Amount
negative_amount = reporting_df["position_amount"] < 0

add_check(
    "Negative Position Amount",
    "Validity",
    negative_amount.sum()
)

# 4. Invalid Maturity Date
invalid_maturity = (
    reporting_df["maturity_date"].notna()
    &
    (
        reporting_df["maturity_date"]
        < reporting_df["origination_date"]
    )
)

add_check(
    "Maturity Before Origination",
    "Consistency",
    invalid_maturity.sum()
)

# 5. Invalid Product Type
invalid_product = ~reporting_df["product_type"].isin(VALID_PRODUCTS)

add_check(
    "Invalid Product Type",
    "Validity",
    invalid_product.sum()
)

# 6. Stale Reporting Date
stale_as_of_date = reporting_df["as_of_date"] != EXPECTED_AS_OF_DATE

add_check(
    "Stale As-Of Date",
    "Timeliness",
    stale_as_of_date.sum()
)

# 7. Duplicate Record IDs
duplicate_records = reporting_df["record_id"].duplicated(keep="first")

add_check(
    "Duplicate Record ID",
    "Uniqueness",
    duplicate_records.sum()
)

In [13]:
pd.DataFrame(dq_summary)

,rule,dimension,failed_records,failure_rate_pct
0,Missing Customer ID,Completeness,100,1.0
1,Invalid Currency,Validity,70,0.7
2,Negative Position Amount,Validity,70,0.7
3,Maturity Before Origination,Consistency,80,0.8
4,Invalid Product Type,Validity,50,0.5
5,Stale As-Of Date,Timeliness,100,1.0
6,Duplicate Record ID,Uniqueness,50,0.5


In [14]:
reporting_unique = (
    reporting_df
    .drop_duplicates(
        subset="record_id",
        keep="first"
    )
)

reconciliation_df = source_df[
    ["record_id", "position_amount"]
].merge(
    reporting_unique[
        ["record_id", "position_amount"]
    ],
    on="record_id",
    suffixes=("_source", "_reporting")
)

reconciliation_df["amount_difference"] = (
    reconciliation_df["position_amount_reporting"]
    - reconciliation_df["position_amount_source"]
).abs()

amount_mismatch = (
    reconciliation_df["amount_difference"] > 0.01
)

add_check(
    "Source-to-Report Amount Mismatch",
    "Accuracy",
    amount_mismatch.sum()
)

print(
    "Source-to-report amount mismatches:",
    amount_mismatch.sum()
)

Source-to-report amount mismatches: 219


In [15]:
dq_summary_df = pd.DataFrame(dq_summary)

dq_summary_df.sort_values(
    "failed_records",
    ascending=False
)

,rule,dimension,failed_records,failure_rate_pct
7,Source-to-Report Amount Mismatch,Accuracy,219,2.18
0,Missing Customer ID,Completeness,100,1.00
5,Stale As-Of Date,Timeliness,100,1.00
3,Maturity Before Origination,Consistency,80,0.80
2,Negative Position Amount,Validity,70,0.70
1,Invalid Currency,Validity,70,0.70
4,Invalid Product Type,Validity,50,0.50
6,Duplicate Record ID,Uniqueness,50,0.50


In [16]:
fig = px.bar(
    dq_summary_df.sort_values(
        "failed_records",
        ascending=True
    ),
    x="failed_records",
    y="rule",
    orientation="h",
    title="Detected Data Quality Issues by Rule",
    labels={
        "failed_records": "Failed Records",
        "rule": "Data Quality Rule"
    },
    hover_data=[
        "dimension",
        "failure_rate_pct"
    ]
)

fig.show()

## 5. Data Quality Scores by Dimension

Individual validation rules are grouped into standard data-quality dimensions to provide an executive-level view of data health.

The dimensions evaluated are:

- Completeness
- Accuracy
- Validity
- Consistency
- Timeliness
- Uniqueness

In [17]:
dimension_scores = (
    dq_summary_df
    .groupby("dimension")
    .agg(
        failed_records=("failed_records", "sum"),
        rules_evaluated=("rule", "count")
    )
    .reset_index()
)

dimension_scores["records_evaluated"] = (
    dimension_scores["rules_evaluated"]
    * len(reporting_df)
)

# Accuracy reconciliation is performed on the unique source/report population.
dimension_scores.loc[
    dimension_scores["dimension"] == "Accuracy",
    "records_evaluated"
] = len(reconciliation_df)

dimension_scores["dq_score_pct"] = (
    100
    - (
        dimension_scores["failed_records"]
        / dimension_scores["records_evaluated"]
        * 100
    )
).round(2)

dimension_scores.sort_values(
    "dq_score_pct",
    ascending=True
)

,dimension,failed_records,rules_evaluated,records_evaluated,dq_score_pct
0,Accuracy,219,1,10000,97.81
1,Completeness,100,1,10050,99.00
3,Timeliness,100,1,10050,99.00
2,Consistency,80,1,10050,99.20
5,Validity,190,3,30150,99.37
4,Uniqueness,50,1,10050,99.50


In [18]:
fig = px.bar(
    dimension_scores.sort_values(
        "dq_score_pct",
        ascending=True
    ),
    x="dq_score_pct",
    y="dimension",
    orientation="h",
    title="Data Quality Score by Dimension",
    labels={
        "dq_score_pct": "Data Quality Score (%)",
        "dimension": "Data Quality Dimension"
    },
    text="dq_score_pct"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_xaxes(
    range=[95, 100]
)

fig.show()

## 6. Overall Data Quality Summary

The individual data-quality dimensions are consolidated into an overall quality view to support management reporting and remediation prioritization.

In [19]:
total_rule_evaluations = int(
    dimension_scores["records_evaluated"].sum()
)

total_failed_checks = int(
    dimension_scores["failed_records"].sum()
)

overall_dq_score = round(
    (
        1
        - total_failed_checks
        / total_rule_evaluations
    )
    * 100,
    2
)

highest_risk_row = dimension_scores.loc[
    dimension_scores["dq_score_pct"].idxmin()
]

kpi_summary = pd.DataFrame(
    {
        "Metric": [
            "Overall Data Quality Score",
            "Reporting Records",
            "Data Quality Rules",
            "Failed Rule Evaluations",
            "Highest-Risk Dimension"
        ],
        "Value": [
            f"{overall_dq_score}%",
            f"{len(reporting_df):,}",
            len(dq_summary_df),
            f"{total_failed_checks:,}",
            highest_risk_row["dimension"]
        ]
    }
)

kpi_summary

,Metric,Value
0,Overall Data Quality Score,99.08%
1,Reporting Records,"10,050"
2,Data Quality Rules,8
3,Failed Rule Evaluations,739
4,Highest-Risk Dimension,Accuracy


### Key Finding

The synthetic reporting dataset demonstrates a high overall data-quality score, while the Accuracy dimension shows the greatest relative weakness. Source-to-report reconciliation therefore represents the highest-priority area for investigation and remediation.

The analysis also demonstrates that aggregate quality scores should not be interpreted in isolation. Individual rule failures, business context, and issue severity must be considered when prioritizing remediation.